In [ ]:
import boto3
from botocore.config import Config
from botocore import UNSIGNED
import io
import polars as pl
import plotly.graph_objects as go
import plotly.subplots as sp
from statsmodels.tsa.seasonal import STL

state = "MN"
upgrade = 0

s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))

files = [ 
    f"up{upgrade:02}-{state.lower()}-mobile_home.csv",
    f"up{upgrade:02}-{state.lower()}-multi-family_with_2_-_4_units.csv",
    f"up{upgrade:02}-{state.lower()}-multi-family_with_5plus_units.csv",
    f"up{upgrade:02}-{state.lower()}-single-family_attached.csv",
    f"up{upgrade:02}-{state.lower()}-single-family_detached.csv"
]
downloads = []
for file in files:
    bucket = "oedi-data-lake"
    key = f"nrel-pds-building-stock/end-use-load-profiles-for-us-building-stock/2024/resstock_tmy3_release_2/timeseries_aggregates/by_state/upgrade={upgrade}/state={state}/{file}"
    
    response = s3.get_object(Bucket=bucket, Key=key)
    buffer = io.BytesIO(response['Body'].read())
    cur = (pl.read_csv(buffer)
                    .select(["timestamp", "units_represented", "out.electricity.total.energy_consumption.kwh", "out.natural_gas.total.energy_consumption.kwh"])
                    .filter(pl.col("timestamp").str.contains("2018-"))
                    .with_columns(pl.col("timestamp").str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S", strict=False)
                                    .dt.offset_by("-15m").dt.replace(year=2025)
                                    .alias("timestamp"))
    )

    downloads.append(cur)

final = (pl.concat(downloads).with_columns(pl.col("timestamp").dt.truncate("1h").alias("timestamp"))
            .group_by(["timestamp"])
            .agg([
                pl.col("units_represented").first().alias("units_represented"),
                pl.col("out.electricity.total.energy_consumption.kwh").sum().alias("electricity.total"),
                pl.col("out.natural_gas.total.energy_consumption.kwh").sum().alias("natural_gas.total")
            ])
        ).sort(["timestamp"])

In [ ]:
import numpy as np
import pandas as pd

stl = STL(final["electricity.total"],period=24,robust=True)
result = stl.fit()

# Subplots: 5 rows, 1 column
fig = sp.make_subplots(rows=5, cols=1, shared_xaxes=True, 
                       subplot_titles=["Original", "Trend", "Seasonal+Residual", "Residual", "adjusted trend"])

fig.add_trace(go.Scatter(x=final["timestamp"], y=final["electricity.total"], name="Original"), row=1, col=1)
fig.add_trace(go.Scatter(x=final["timestamp"], y=result.trend, name="Trend"), row=2, col=1)
fig.add_trace(go.Scatter(x=final["timestamp"], y=result.seasonal+result.resid, name="Seasonal+Residual"), row=3, col=1)
fig.add_trace(go.Scatter(x=final["timestamp"], y=result.resid, name="Residual"), row=4, col=1)

# 1. Extract original components
trend = result.trend.copy()
seasonal_resid = result.seasonal + result.resid

# 2. Define a mask for winter months
winter_mask = final["timestamp"].to_pandas().dt.month.isin([10,11,12,1,2,3,4,5])

# 3. Define a scaling factor for winter trend (e.g., 80%)
scale = 0.55

# 4. Optionally apply a smooth taper in/out (e.g., via rolling blend)
blend = np.ones_like(trend)
blend[winter_mask] = scale

# Optional: apply smoothing to blend (e.g., 7-day rolling to avoid hard edges)
blend = pd.Series(blend).rolling(window=24*30, center=True, min_periods=1).mean()

# 5. Apply blended scale to trend
adjusted_trend = trend * blend

fig.add_trace(go.Scatter(x=final["timestamp"], y=adjusted_trend, name="adjusted trend"), row=5, col=1)

fig.update_layout(height=1000, title="STL Decomposition", showlegend=False)
fig.show()